# Phase 1 — the data authorities

---

zlabel labels one scRNA-seq cluster from its marker genes. Before it can score or ground anything, it needs three **data authorities** loaded into memory. This notebook is both a **walkthrough of Phase 1** and a **reusable exploration tool** for those authorities.

The full annotation loop (`docs/design.md`) is:
1. **Normalize** symbols
2. **Score** against panels (a coarse prior)
3. **Converge** on ZFA anatomy (the namer)
4. **Guardrail** against the panel's ontology anchor
5. **Decide**
6. **Emit** a `Label` packet

**Phase 1 implements none of those steps.** It builds the substrate they all read:

| Authority | Loader | What it gives you |
|---|---|---|
| **ZFA** anatomy graph | `load_zfa` | a DAG of anatomy terms + an ancestor walk |
| **ZFIN** wildtype expression | `load_zfin_expression` | where each gene was seen *in vivo* (ZFA + stage) |
| **GAF** gene synonyms | `load_gene_synonym_map` | any alias / old name → current symbol |

Each section follows the same rhythm: **context → load → look at it**. We render every structured output as a [rich](https://rich.readthedocs.io) table or tree so the shape of the data is legible at a glance, and we keep the interactive pyvis graph at the end (Optional).

<div class="alert alert-warning" role="alert">
    <b>⚙️ Prerequisite — run before this notebook</b>
    <br><br>
    <b>1.</b> From the repo root: <code>bash scripts/setup_data.sh</code> &nbsp;(downloads the ontology files into <code>data/ontologies/</code>)
    <br>
    <b>2.</b> Start the server with <code>make notebook</code> &nbsp;(keeps the working directory at the repo root)
</div>

In [1]:
# --- Bootstrap: locate the package root and the downloaded ontology files ------
import zlabel
import os
from pathlib import Path

print(f"zlabel {zlabel.__version__}")

# DATA_DIR is resolved relative to CWD, which is the repo root when the server is
# started with `make notebook`. We split the CWD on the package name so the path
# is correct regardless of how deep inside the repo the kernel happens to start.
PACKAGE = "zlabel"
ROOT_DIR = Path(os.getcwd().split(PACKAGE, 1)[0]) / PACKAGE
DATA_DIR = ROOT_DIR / Path("data/ontologies")
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data not found at {DATA_DIR.resolve()}.  Run: bash scripts/setup_data.sh"
    )
print(f"data at {DATA_DIR.resolve()}")

# --- rich: one Console for the whole notebook ---------------------------------
# Every structured output below is rendered with rich so the *shape* of the data
# reads at a glance. Tables for tabular data, Tree for ZFA ancestor chains.
from rich.console import Console
from rich.table import Table
from rich.tree import Tree

console = Console()

# Color key (rich style strings) — used consistently throughout the notebook:
#   green = resolved / ok        yellow = ambiguous / warning
#   red   = unresolved / error   dim    = zero / empty
OK, WARN, ERR, DIM = "green", "yellow", "red", "dim"

zlabel 0.1.0
data at /home/ec2-user/PycharmProjects/zlabel/data/ontologies


## 1. ZFA — Zebrafish Anatomy Ontology

The ZFA is a directed graph of anatomical terms: a **term** is any named structure (e.g. `ZFA:0000014` — *dorsal aorta*), and a **directed edge** points from a more specific term to a broader one, labelled by the relationship that connects them.

```
 ──────────────────────────────────────────────────────────────────────────────────────
|  dorsal aorta  ──is_a──▶  artery  ──part_of──▶  trunk vasculature  ──...──▶  ...        |
 ──────────────────────────────────────────────────────────────────────────────────────
```

### Format: what `load_zfa` reads

The source is a `.obo` file — one `[Term]` stanza per structure. `load_zfa` hands it to `obonet`, which turns each stanza into a graph node and each relationship line into a child → parent edge. The fields that matter to us:

```text
[Term]
id: ZFA:0000014                                  ← node id
name: dorsal aorta                               ← term_name() returns this
is_a: ZFA:0000005 ! artery                       ← edge, key "is_a"        (a kind-of)
relationship: part_of ZFA:0005024 ! trunk vasculature   ← edge, key "part_of"   (a part-of)
relationship: develops_from ZFA:0005077 ! vascular cord ← edge, key "develops_from" (lineage)
```

`load_zfa` returns a `networkx.MultiDiGraph`: nodes are ZFA IDs, edges go child → parent **keyed by relationship type**. The MultiDiGraph (not a plain DiGraph) matters because one pair of terms can be joined by more than one relationship.

### The `ancestors` BFS walk (lightly unfolded)

`ancestors` does a breadth-first walk *up* the graph and returns **all transitive ancestors, nearest-first** — not just immediate parents, but everything reachable by following the chosen edge types:

```text
seen ← {start}                         # seed with start so it can't re-appear as its own ancestor
queue ← [start]
while queue:
    node ← queue.popleft()
    for (node → parent) keyed by rel:  # out-edges = parents, in a child→parent graph
        if rel in allowed and parent not in seen:
            record parent; enqueue parent
```

Two edge-type choices give two views of the same structure:

- **`is_a + part_of` (default)** — structural lineage: what category, and what larger structure does it belong to? dorsal aorta → artery → trunk vasculature → … → cardiovascular system.
- **`develops_from`** — developmental lineage: what embryonic precursor did this arise from? dorsal aorta → vascular cord. Useful when the question is cell fate, not mature anatomy.

**Why this matters for the labeller:** a gene annotated as expressing *"in dorsal aorta"* implicitly expresses in every ancestor too. Walking the graph propagates that evidence upward — the mechanism the Phase 4 convergence vote stands on.

**What to look for:** four terms contain *"aorta"*; the **dorsal aorta** ancestor chain climbs from a specific artery all the way to *whole organism* (default `is_a + part_of`), while the `develops_from`-only view is short — a single embryonic precursor. Same term, two lineages.

In [2]:
# --- Load the ZFA anatomy graph ------------------------------------------------
zfa_ontology = zlabel.load_zfa(DATA_DIR / "zfa.obo")
console.print(
    f"[bold]ZFA[/bold]: {len(zfa_ontology.nodes):,} terms, {len(zfa_ontology.edges):,} edges"
)

# --- Search the graph for every term whose name contains "aorta" ---------------
# The walrus (:=) assigns term_name() while filtering, so it's called once per node.
aorta_hits = sorted(
    (
        (nid, name)
        for nid in zfa_ontology.nodes
        if (name := zlabel.term_name(zfa_ontology, nid)) and "aorta" in name
    ),
    key=lambda pair: pair[1],
)

# Render the hits as a rich table (ZFA id, name).
hits_table = Table(title="ZFA terms containing 'aorta'", title_style="bold")
hits_table.add_column("ZFA id", style="cyan", no_wrap=True)
hits_table.add_column("name")
for term_id, name in aorta_hits:
    hits_table.add_row(term_id, name)
console.print(hits_table)

# --- Pick dorsal aorta as a stable example term --------------------------------
# tid / name are reused by the optional pyvis cell at the end of the notebook.
tid, name = next(
    ((nid, nm) for nid, nm in aorta_hits if nm == "dorsal aorta"),
    aorta_hits[0],
)

# --- Walk the ancestors two ways and render each as a rich.Tree ----------------
# Default (is_a + part_of): the structural lineage, nearest-first.
structural = zlabel.ancestors(zfa_ontology, tid)
tree = Tree(f"[bold cyan]{tid}[/bold cyan]  {name}  [dim](is_a + part_of)[/dim]")
branch = tree
for parent_id in structural:  # nearest-first → nest each ancestor under the previous
    branch = branch.add(f"[cyan]{parent_id}[/cyan]  {zlabel.term_name(zfa_ontology, parent_id)}")
console.print(tree)

# Lineage-only (develops_from): the embryonic precursor chain — typically far shorter.
lineage = zlabel.ancestors(zfa_ontology, tid, edge_types={"develops_from"})
lineage_tree = Tree(f"[bold cyan]{tid}[/bold cyan]  {name}  [dim](develops_from only)[/dim]")
if lineage:
    branch = lineage_tree
    for parent_id in lineage:
        branch = branch.add(
            f"[cyan]{parent_id}[/cyan]  {zlabel.term_name(zfa_ontology, parent_id)}"
        )
else:
    lineage_tree.add(f"[{DIM}](none via develops_from)[/{DIM}]")
console.print(lineage_tree)

ZFA: 3,161 terms, 12,293 edges

         ZFA terms containing 'aorta'         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ZFA id      ┃ name                         ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ZFA:0000014 │ dorsal aorta                 │
│ ZFA:0001054 │ lateral dorsal aorta         │
│ ZFA:0000604 │ ventral aorta                │
│ ZFA:0005028 │ ventral wall of dorsal aorta │
└─────────────┴──────────────────────────────┘

ZFA:0000014  dorsal aorta  (is_a + part_of)
└── ZFA:0000005  artery
    └── ZFA:0005024  trunk vasculature
        └── ZFA:0005295  axial blood vessel
            └── ZFA:0005314  blood vessel
                └── ZFA:0001079  blood vasculature
                    └── ZFA:0005249  vasculature
                        └── ZFA:0001115  trunk
                            └── ZFA:0001073  axial vasculature
                                └── ZFA:0001488  multi-tissue structure
                                    └── ZFA:0001490  cavitated compound organ
                                        └── ZFA:0000010  cardiovascular system
                                            └── ZFA:0001308  organism subdivision
                                                └── ZFA:0000037  anatomical structure
                                                    └── ZFA:0000496  compound organ
                                                        └── ZFA:0001439  anatomical system
                                                            └── ZFA:0001094  whole organism
                                                                └── ZFA:0100000  zebrafish anatomical entity
                                                                    └── ZFA:0001512  anatomical group

ZFA:0000014  dorsal aorta  (develops_from only)
└── ZFA:0005077  vascular cord

## 2. ZFIN wildtype expression — the keystone of Phase 1

This is the loader to understand. `load_zfin_expression` parses the curated ZFIN wildtype-expression file — every observation of *"this gene expresses in this anatomy at this stage, in normal fish"* — into a dict keyed by **lowercased gene symbol**. Each value is a list of `ZfinExpressionRecord` objects, one per observation. These are the **in-vivo grounding signals**: where do a cluster's markers actually express?

### The file format

Tab-delimited, **no header, 15 columns**. The loader reads only six of them:

| 0-idx | col | field | used as |
|---|---|---|---|
| 1 | 2 | gene symbol | dict key (lowercased) |
| 3 | 4 | **super**-structure ZFA id | anatomy (coarser) |
| 4 | 5 | super-structure name | |
| 5 | 6 | **sub**-structure ZFA id | anatomy (finer) |
| 6 | 7 | sub-structure name | |
| 7 | 8 | ZFS start stage | stage window start |
| 8 | 9 | ZFS end stage | stage window end |

Note the stages are **ZFS names, not hours** — `Long-pec`, `Larval:Day 4`, `Adult`.

### The non-obvious rule: keep the *most specific* structure

A single ZFIN row can name **two** anatomy terms: a broad super-structure *and* a narrower sub-structure inside it. For example, one `kdrl` row records super = *anterior segment eye* and sub = *artery* — the gene was seen in an artery, which lives in the eye.

`_most_specific_structure` keeps the **sub-structure when the row has one, else the super-structure**. The finer term is the better grounding, and nothing is lost: the coarser ancestors are still recoverable later by the ZFA `ancestors` walk from Section 1. So `load_zfin_expression` produces, per gene, the union of the most-specific structure from every one of its rows.

The next cell reads a few **raw lines** and reproduces that extraction by hand, so you can see exactly what the loader builds.

**What to look for:** the second row carries **both** a super- and a sub-structure; the "most specific" column keeps the **sub** (`artery`), discarding the coarser `anterior segment eye`. The first row has only a super-structure, so that one is kept. This hand-extraction reproduces exactly what `load_zfin_expression` stores.

In [3]:
# --- Reproduce the loader's extraction on a few RAW lines ----------------------
# Read kdrl rows straight from the file and apply the same column rules by hand,
# so the "most specific structure" decision is visible. These 0-indexed positions
# match the _COL_* constants in src/zlabel/data.py exactly.
EXPR_FILE = DATA_DIR / "zfin_wildtype_expression.txt"
COL_SYMBOL, COL_SUPER_ID, COL_SUPER_NAME = 1, 3, 4
COL_SUB_ID, COL_SUB_NAME, COL_START, COL_END = 5, 6, 7, 8

# Grab one super-only kdrl row and one super+sub kdrl row to show both branches
# of the rule. (We scan for the first of each; the file is sorted by gene.)
super_only = None
super_and_sub = None
with EXPR_FILE.open(encoding="utf-8") as handle:
    for line in handle:
        fields = [f.strip() for f in line.split("\t")]
        if len(fields) <= COL_END or fields[COL_SYMBOL] != "kdrl":
            continue
        if fields[COL_SUB_ID] and super_and_sub is None:
            super_and_sub = fields
        elif not fields[COL_SUB_ID] and super_only is None:
            super_only = fields
        if super_only and super_and_sub:
            break

# The exact rule from data._most_specific_structure: sub wins when present.
def most_specific(fields: list[str]) -> tuple[str, str]:
    """Return (zfa_id, zfa_name): the sub-structure if the row has one, else the super."""
    if fields[COL_SUB_ID]:
        return fields[COL_SUB_ID], fields[COL_SUB_NAME]
    return fields[COL_SUPER_ID], fields[COL_SUPER_NAME]

# Render the raw → extracted mapping as a rich table.
raw_table = Table(title="Raw kdrl rows → most-specific structure (by hand)", title_style="bold")
raw_table.add_column("super (col 4/5)", style="dim")
raw_table.add_column("sub (col 6/7)")
raw_table.add_column("→ kept (most specific)", style=OK)
raw_table.add_column("stage window", style="cyan")
for fields in (super_only, super_and_sub):
    super_cell = f"{fields[COL_SUPER_ID]}  {fields[COL_SUPER_NAME]}"
    sub_cell = (
        f"{fields[COL_SUB_ID]}  {fields[COL_SUB_NAME]}"
        if fields[COL_SUB_ID]
        else f"[{DIM}](none)[/{DIM}]"
    )
    kept_id, kept_name = most_specific(fields)
    raw_table.add_row(
        super_cell,
        sub_cell,
        f"{kept_id}  {kept_name}",
        f"{fields[COL_START]} .. {fields[COL_END]}",
    )
console.print(raw_table)

# --- Confirm the by-hand result matches the library loader ---------------------
expression_map = zlabel.load_zfin_expression(EXPR_FILE)
hand_extracted = {most_specific(super_only), most_specific(super_and_sub)}
loader_structures = {(r.zfa_id, r.zfa_name) for r in expression_map["kdrl"]}
assert hand_extracted <= loader_structures, "hand extraction should be a subset of the loader's"
console.print(
    f"[{OK}]✓[/{OK}] both hand-extracted structures appear in "
    f"load_zfin_expression()['kdrl'] ({len(loader_structures)} unique structures total)"
)

                                 Raw kdrl rows → most-specific structure (by hand)                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ super (col 4/5)              ┃ sub (col 6/7)       ┃ → kept (most specific)       ┃ stage window                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ZFA:0009258  angioblastic    │ (none)              │ ZFA:0009258  angioblastic    │ Segmentation:10-13 somites  │
│ mesenchymal cell             │                     │ mesenchymal cell             │ .. Segmentation:10-13       │
│                              │                     │                              │ somites                     │
│ ZFA:0005566  anterior        │ ZFA:0000005  artery │ ZFA:0000005  artery          │ Larval:Day 4 .. Larval:Day  │
│ segment eye                  │                     │                              │ 4                           │
└──────────────────────────────┴─────────────────────┴──────────────────────────────┴─────────────────────────────┘

✓ both hand-extracted structures appear in load_zfin_expression()['kdrl'] (75 unique structures total)

### The whole map, per gene

Now the same `expression_map` viewed by gene. Three canonical markers from different broad lineages — *kdrl* (endothelium), *gata1a* (blood), *sox17* (endoderm) — each carries hundreds of records collapsing onto dozens of unique structures.

**What to look for:** the **record count** column shows how often each structure was observed — repeated observations are stronger evidence. A gene's footprint is broad (many structures) but the high-count rows concentrate in its true lineage (*gata1a* → blood, blood island).

In [4]:
# --- Per-gene structure counts -------------------------------------------------
# expression_map was loaded in the cell above; re-state it here so this section
# reads standalone. Counter tallies how many records fall on each ZFA structure.
from collections import Counter

console.print(
    f"[bold]ZFIN expression[/bold]: {len(expression_map):,} genes with curated records"
)

# Three canonical markers spanning different broad lineages.
counts_table = Table(title="Per-gene expression structures (top 6 each)", title_style="bold")
counts_table.add_column("gene", style="magenta", no_wrap=True)
counts_table.add_column("ZFA id", style="cyan", no_wrap=True)
counts_table.add_column("structure")
counts_table.add_column("records", justify="right", style=OK)

for gene in ("kdrl", "gata1a", "sox17"):
    records = expression_map.get(gene, [])
    structure_counts = Counter((r.zfa_id, r.zfa_name) for r in records)

    # One header-ish summary row per gene, then its top structures by name.
    counts_table.add_row(
        f"[bold]{gene}[/bold]",
        "",
        f"[{DIM}]{len(records)} records · {len(structure_counts)} unique structures[/{DIM}]",
        "",
    )
    top = sorted(structure_counts.items(), key=lambda item: item[0][1])[:6]
    for (zfa_id, zfa_name), n in top:
        counts_table.add_row("", zfa_id, zfa_name, str(n))
    if len(structure_counts) > 6:
        counts_table.add_row("", "", f"[{DIM}]… {len(structure_counts) - 6} more[/{DIM}]", "")
    counts_table.add_section()

console.print(counts_table)

ZFIN expression: 14,485 genes with curated records

              Per-gene expression structures (top 6 each)               
┏━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ gene   ┃ ZFA id       ┃ structure                          ┃ records ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ kdrl   │              │ 482 records · 75 unique structures │         │
│        │ ZFA:0009258  │ angioblastic mesenchymal cell      │       8 │
│        │ ZFA:0005604  │ angiogenic sprout                  │       5 │
│        │ ZFA:0005039  │ anterior lateral mesoderm          │       3 │
│        │ ZFA:0005041  │ anterior lateral plate mesoderm    │       5 │
│        │ ZFA:0005004  │ aortic arch                        │       2 │
│        │ ZFA:0000005  │ artery                             │       2 │
│        │              │ … 69 more                          │         │
├────────┼──────────────┼────────────────────────────────────┼─────────┤
│ gata1a │              │ 529 records · 50 unique structures │         │
│        │ ZFA:0005039  │ anterior lateral mesoderm          │       3 │
│        │ ZFA:0001073  │ axial vasculature                  │       1 │
│        │ ZFA:0000006  │ ball                               │       2 │
│        │ ZFA:0000007  │ blood                              │      14 │
│        │ ZFA:0009044  │ blood cell                         │       3 │
│        │ ZFA:0000094  │ blood island                       │      13 │
│        │              │ … 44 more                          │         │
├────────┼──────────────┼────────────────────────────────────┼─────────┤
│ sox17  │              │ 184 records · 20 unique structures │         │
│        │ ZFA:0000001  │ Kupffer's vesicle                  │       3 │
│        │ ZFA:0005041  │ anterior lateral plate mesoderm    │       1 │
│        │ ZFA:0001176  │ blastoderm                         │       1 │
│        │ ZFA:0001175  │ blastodisc                         │       1 │
│        │ ZFA:0000186  │ common cardinal vein               │       1 │
│        │ BSPO:0000079 │ dorsal region                      │       1 │
│        │              │ … 14 more                          │         │
└────────┴──────────────┴────────────────────────────────────┴─────────┘

## 3. GAF gene synonyms

Zebrafish genes carry aliases, deprecated symbols, and `a`/`b` paralog names; a real marker can be missed purely because a dataset used an old symbol. `load_gene_synonym_map` builds the fix — an **alias → current-symbol** lookup — by inverting the GAF's synonym column.

### Format: two columns of a GAF line

The GAF is tab-delimited GO annotations. The loader reads only two columns (0-indexed `_GAF_COL_SYMBOL = 2`, `_GAF_COL_SYNONYMS = 10` in `data.py`):

```text
col 3  (DB Object Symbol)  = kdrl                          ← the current symbol
col 11 (DB Object Synonym) = VEGFR-2|flk1|kdr|vegfr2|...   ← pipe-separated aliases + old names
```

### The inversion: synonym → current symbol

The file lists `current → [aliases]`; the labeller needs the reverse. So for every row the loader maps **the current symbol to itself, and each pipe-separated alias to that symbol**, case-folded:

```text
kdrl   → {kdrl}
flk1   → {kdrl}      vegfr2 → {kdrl}      kdr → {kdr, kdrl}   ← old name shared by two genes
```

One previous-name can fan out to **several** current paralogs (that's the `set` value), which is exactly the ambiguity `genes.normalize_symbol` (Phase 2) must surface rather than guess through.

`genes.normalize_symbol` consumes this map as step 1 of the annotation loop — so a marker submitted as `kdr` or `flk1` still resolves to `kdrl`.

**What to look for:** the status colors tell the story. **Green** = a clean single resolution (`flk1 → kdrl`). **Yellow** = an old name that fans out to several current paralogs (`dermo1 → twist2, twist3`) — the map reports the ambiguity, it does not pick. **Red** = a name absent from the map. Note `kdr` resolves to itself even though it is *also* an old alias of `kdrl`: an input that is a current symbol keeps its own identity (Phase 2's rule, previewed here).

In [5]:
# --- Build the synonym map -----------------------------------------------------
synonym_map = zlabel.load_gene_synonym_map(DATA_DIR / "zfin.gaf")
console.print(
    f"[bold]GAF synonym map[/bold]: {len(synonym_map):,} entries (aliases + current symbols)"
)


# --- Classify each probe the way Phase 2's normalize_symbol will -------------
# This mirrors genes.normalize_symbol's three outcomes so the status colors here
# match what Phase 2 reports. (Phase 2 does the real thing; we preview it.)
def classify(name: str) -> tuple[str, set[str], str]:
    """Return (status, resolved_symbols, note) for a probe against synonym_map."""
    key = name.lower()
    candidates = synonym_map.get(key)
    if candidates is None:
        return "unresolved", set(), "not in GAF synonym map"
    # An input that is itself a current symbol keeps its own identity.
    if key in candidates:
        return "resolved", {key}, "is a current symbol"
    if len(candidates) == 1:
        return "resolved", set(candidates), "single alias target"
    # A previous name that fans out to multiple current paralogs is ambiguous.
    return "ambiguous", set(candidates), f"fans out to {len(candidates)} paralogs"


# Probes chosen to hit all three statuses: clean aliases, a self-resolving
# current symbol (kdr), a genuine paralog fan-out (dermo1, pvalb), and a miss.
probes = ["kdrl", "flk1", "vegfr2", "kdr", "gata1", "dermo1", "pvalb", "notreal_xyz"]
STATUS_STYLE = {"resolved": OK, "ambiguous": WARN, "unresolved": ERR}

probe_table = Table(title="Synonym resolution probes", title_style="bold")
probe_table.add_column("input", style="magenta", no_wrap=True)
probe_table.add_column("status", no_wrap=True)
probe_table.add_column("resolved to")
probe_table.add_column("note", style=DIM)
for alias in probes:
    status, resolved, note = classify(alias)
    style = STATUS_STYLE[status]
    resolved_cell = ", ".join(sorted(resolved)) if resolved else f"[{DIM}](none)[/{DIM}]"
    probe_table.add_row(alias, f"[{style}]{status}[/{style}]", resolved_cell, note)
console.print(probe_table)

GAF synonym map: 66,682 entries (aliases + current symbols)

                      Synonym resolution probes                       
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ input       ┃ status     ┃ resolved to    ┃ note                   ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ kdrl        │ resolved   │ kdrl           │ is a current symbol    │
│ flk1        │ resolved   │ kdrl           │ single alias target    │
│ vegfr2      │ resolved   │ kdrl           │ single alias target    │
│ kdr         │ resolved   │ kdr            │ is a current symbol    │
│ gata1       │ resolved   │ gata1a         │ single alias target    │
│ dermo1      │ ambiguous  │ twist2, twist3 │ fans out to 2 paralogs │
│ pvalb       │ ambiguous  │ pvalb2, pvalb7 │ fans out to 2 paralogs │
│ notreal_xyz │ unresolved │ (none)         │ not in GAF synonym map │
└─────────────┴────────────┴────────────────┴────────────────────────┘

## 4. Explore it yourself

The three authorities are loaded into `zfa_ontology`, `expression_map`, and `synonym_map`. The two helpers below turn this notebook into a **reusable probe** over them — point them at any gene or ZFA term.

- `explore_gene(symbol)` — every ZFIN expression record for a gene (ZFA id, structure, stage window).
- `explore_term(zfa_id)` — a term's name, its ancestor chain (rich.Tree), and a few direct children.

In [6]:
# --- Reusable exploration helpers ----------------------------------------------
from collections import Counter as _Counter

from zlabel.data import DEFAULT_ANCESTOR_EDGE_TYPES


def explore_gene(symbol: str, max_rows: int = 20) -> None:
    """Print a gene's ZFIN expression records as a rich table.

    Args:
        symbol (str): Gene symbol, any case (expression_map keys are lowercased).
        max_rows (int): Cap on rows shown, longest-window structures first.
    """
    records = expression_map.get(symbol.lower(), [])
    if not records:
        console.print(f"[{ERR}]no ZFIN expression records for {symbol!r}[/{ERR}]")
        return
    # Collapse to one row per structure with its observation count.
    by_structure = _Counter((r.zfa_id, r.zfa_name) for r in records)
    table = Table(
        title=f"{symbol} — {len(records)} records, {len(by_structure)} structures",
        title_style="bold",
    )
    table.add_column("ZFA id", style="cyan", no_wrap=True)
    table.add_column("structure")
    table.add_column("records", justify="right", style=OK)
    table.add_column("a stage window", style="dim")
    # One example stage window per structure (records list keeps insertion order).
    example_stage = {(r.zfa_id, r.zfa_name): f"{r.start_stage} .. {r.end_stage}" for r in records}
    for (zfa_id, zfa_name), n in by_structure.most_common(max_rows):
        table.add_row(zfa_id, zfa_name, str(n), example_stage[(zfa_id, zfa_name)])
    if len(by_structure) > max_rows:
        table.add_row("", f"[{DIM}]… {len(by_structure) - max_rows} more[/{DIM}]", "", "")
    console.print(table)


def explore_term(zfa_id: str, max_children: int = 8) -> None:
    """Print a ZFA term's name, ancestor Tree, and a few direct children.

    Args:
        zfa_id (str): OBO term id, e.g. ZFA:0000014.
        max_children (int): Cap on direct children listed.
    """
    if zfa_id not in zfa_ontology:
        console.print(f"[{ERR}]{zfa_id!r} not in the ZFA graph[/{ERR}]")
        return
    name = zlabel.term_name(zfa_ontology, zfa_id) or "(unnamed)"

    # Ancestors (default is_a + part_of), nearest-first, as a nested Tree.
    tree = Tree(f"[bold cyan]{zfa_id}[/bold cyan]  {name}")
    branch = tree
    for parent_id in zlabel.ancestors(zfa_ontology, zfa_id):
        branch = branch.add(f"[cyan]{parent_id}[/cyan]  {zlabel.term_name(zfa_ontology, parent_id)}")
    console.print(tree)

    # Direct children: an in-edge keyed is_a/part_of means "child → this term".
    children = [
        child
        for child, _, rel in zfa_ontology.in_edges(zfa_id, keys=True)
        if rel in DEFAULT_ANCESTOR_EDGE_TYPES
    ]
    if children:
        console.print(f"[{DIM}]direct children ({len(children)}):[/{DIM}]")
        for child_id in children[:max_children]:
            console.print(f"  [cyan]{child_id}[/cyan]  {zlabel.term_name(zfa_ontology, child_id)}")
    else:
        console.print(f"[{DIM}](leaf — no is_a/part_of children)[/{DIM}]")


console.print(f"[{OK}]ready[/{OK}] — call explore_gene('...') or explore_term('ZFA:...')")

ready — call explore_gene('...') or explore_term('ZFA:...')

**Now try your own.** Swap in any gene symbol or ZFA id below. Good starting points: genes `myod1`, `cdh5`, `mpx`; terms `ZFA:0000007` (blood), `ZFA:0000114` (heart), `ZFA:0009065` (endothelial cell).

In [7]:
# --- Try the helpers — edit these two values and re-run -------------------------
explore_gene("cdh5")        # an endothelial cadherin; try myod1, mpx, sox17, ...
explore_term("ZFA:0000114")  # heart; try ZFA:0000007 (blood), ZFA:0009065 (endothelial cell), ...

                                         cdh5 — 188 records, 68 structures                                         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ZFA id      ┃ structure                                  ┃ records ┃ a stage window                             ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ZFA:0009065 │ endothelial cell                           │      13 │ Adult .. Adult                             │
│ ZFA:0000014 │ dorsal aorta                               │      12 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0001285 │ intersegmental vessel                      │      12 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0000477 │ posterior cardinal vein                    │       9 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0009036 │ blood vessel endothelial cell              │       9 │ Segmentation:20-25 somites ..              │
│             │                                            │         │ Segmentation:20-25 somites                 │
│ ZFA:0001079 │ blood vasculature                          │       8 │ Segmentation:26+ somites ..                │
│             │                                            │         │ Segmentation:26+ somites                   │
│ ZFA:0001094 │ whole organism                             │       8 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0001320 │ endocardium                                │       6 │ Segmentation:26+ somites ..                │
│             │                                            │         │ Segmentation:26+ somites                   │
│ ZFA:0001093 │ unspecified                                │       6 │ Segmentation:5-9 somites ..                │
│             │                                            │         │ Segmentation:5-9 somites                   │
│ ZFA:0005024 │ trunk vasculature                          │       5 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0000186 │ common cardinal vein                       │       5 │ Segmentation:26+ somites ..                │
│             │                                            │         │ Segmentation:26+ somites                   │
│ ZFA:0005249 │ vasculature                                │       5 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0001073 │ axial vasculature                          │       4 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0000008 │ brain                                      │       4 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0000423 │ anterior cardinal vein                     │       3 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0001054 │ lateral dorsal aorta                       │       3 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0001639 │ vascular endothelium                       │       3 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│ ZFA:0001616 │ atrioventricular canal endocardium         │       3 │ Larval:Protruding-mouth ..                 │
│             │                                            │         │ Larval:Protruding-mouth                    │
│ ZFA:0001051 │ caudal division of the internal carotid    │       3 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│             │ artery                                     │         │                                            │
│ ZFA:0001267 │ cranial vasculature                        │       3 │ Pharyngula:Prim-5 .. Pharyngula:Prim-5     │
│             │ … 48 more                                  │         │                                            │
└─────────────┴────────────────────────────────────────────┴─────────┴────────────────────────────────────────────┘

ZFA:0000114  heart
└── ZFA:0001490  cavitated compound organ
    └── ZFA:0000010  cardiovascular system
        └── ZFA:0007115  pericardial region
            └── ZFA:0000496  compound organ
                └── ZFA:0001439  anatomical system
                    └── ZFA:0001478  anatomical cluster
                        └── ZFA:0000037  anatomical structure
                            └── ZFA:0001094  whole organism
                                └── ZFA:0001512  anatomical group
                                    └── ZFA:0100000  zebrafish anatomical entity

direct children (12):

ZFA:0000009  cardiac ventricle

ZFA:0000154  sinus venosus

...

ZFA:0000471  atrium

ZFA:0001315  atrioventricular canal

ZFA:0001319  myocardium

ZFA:0001320  endocardium

ZFA:0005057  epicardium

## 5. Synthesis — the Phase 1 handoff

Phase 1 produces exactly **three in-memory artifacts**, and every later phase reads them:

| Artifact | Type | Built by | Consumed by |
|---|---|---|---|
| `zfa_ontology` | `networkx.MultiDiGraph` | `load_zfa` | the ZFA convergence vote (Phase 4) walks ancestors here |
| `expression_map` | `dict[str, list[ZfinExpressionRecord]]` | `load_zfin_expression` | grounding (Phase 3) — *where does this marker express?* |
| `synonym_map` | `dict[str, set[str]]` | `load_gene_synonym_map` | `normalize_symbol` (Phase 2) — *what is this marker's current name?* |

No labeling logic lives in Phase 1 — these loaders only turn files into data structures. The three together are the **substrate** the annotation loop stands on; everything downstream is composition over them.

## 6. What's next

Continue with the [Phase 2 notebook](phase_02.ipynb), which feeds `synonym_map` into `normalize_symbol` (the resolved / ambiguous / unresolved split previewed above) and scores normalized markers against curated panel buckets.

Then:
- **Phase 3** wires it together: normalized markers → panel scores → ZFIN grounding → converging-evidence decision → `Label` packet.
- **Phase 4a** makes the label's name and depth come from a ZFA convergence vote over `expression_map` + `zfa_ontology` (panels demote to the coarse prior + guardrail).
- **Phase 4b** measures the whole loop against the Daniocell atlas.

<div class="alert alert-success" role="alert">
    <b>✅ What you have after Phase 1</b>
    <br><br>
    Three loaded authorities — <code>zfa_ontology</code> (anatomy graph), <code>expression_map</code> (in-vivo grounding), and <code>synonym_map</code> (alias resolver) — plus two reusable probes, <code>explore_gene</code> and <code>explore_term</code>.
</div>

## Appendix (Optional) — interactive ZFA neighborhood

*This cell is optional and is not part of the core top-to-bottom path.* The rich `Tree` above already renders the dorsal-aorta ancestor chain as text; this pyvis graph is the same data, **augmented** with an interactive, draggable, color-coded view. Skip it if you only need the analysis.

**What to look for:** the selected term (dorsal aorta) is highlighted in **red**, and parent edges are color-coded by relationship — `is_a` (green), `part_of` (blue), `develops_from` (yellow) — the same three edge types the `ancestors` walk follows.

In [ ]:
# --- Optional interactive ZFA neighborhood (pyvis) -----------------------------
# Renders a term + its ancestors as a draggable hierarchical graph. This AUGMENTS
# the rich Tree above; it does not replace any analysis. Keep it self-contained so
# it still runs if this cell is executed on its own.
import networkx as nx
from IPython.display import HTML, display
from pyvis.network import Network

# Edge colors match the markdown legend: is_a=green, part_of=blue, develops_from=yellow.
_EDGE_COLOURS = {
    "is_a": "#A6E3A1",
    "part_of": "#89B4FA",
    "develops_from": "#F9E2AF",
}


def show_subgraph_interactive(
    graph: nx.MultiDiGraph,
    term_id: str | None = None,
) -> None:
    """Draw a term and its is_a/part_of ancestors as an interactive pyvis graph."""
    # Restrict to the term + its ancestors (a readable neighborhood), or the whole graph.
    if term_id:
        ancestor_ids = zlabel.ancestors(graph, term_id)
        subg = graph.subgraph([term_id, *ancestor_ids])
    else:
        subg = graph

    net = Network(
        height="800px",
        width="100%",
        directed=True,
        bgcolor="#1e1e2e",
        font_color="#cdd6f4",
        notebook=True,
        cdn_resources="in_line",  # embeds vis.js directly — avoids CDN failures in Jupyter
    )

    # Nodes: the focus term is red, everything else blue.
    for nid in subg.nodes:
        label = zlabel.term_name(graph, nid) or nid
        colour = "#E84C4C" if nid == term_id else "#4C9BE8"
        net.add_node(nid, label=label, title=f"{nid}\n{label}", color=colour, size=12)

    # Edges: colored by relationship key (is_a / part_of / develops_from).
    for u, v, key in subg.edges(keys=True):
        net.add_edge(u, v, title=key, color=_EDGE_COLOURS.get(key, "#666"), width=1.5)

    net.set_options("""{
      "layout": {"hierarchical": {"enabled": true, "direction": "UD", "sortMethod": "directed"}},
      "physics": {"enabled": false}
    }""")

    display(HTML(net.generate_html()))


# Reconstruct zfa_ontology / tid if this optional cell is run before the rest.
if "zfa_ontology" not in globals():
    zfa_ontology = zlabel.load_zfa(DATA_DIR / "zfa.obo")
if "tid" not in globals():
    aorta_hits = sorted(
        (
            (nid, name)
            for nid in zfa_ontology.nodes
            if (name := zlabel.term_name(zfa_ontology, nid)) and "aorta" in name
        ),
        key=lambda pair: pair[1],
    )
    tid = next((nid for nid, nm in aorta_hits if nm == "dorsal aorta"), aorta_hits[0][0])

print("Rendering optional ZFA neighborhood graph...")
show_subgraph_interactive(zfa_ontology, tid)